# Text Embeddings, Similarity, and Retrieval Chatbot

This notebook helps students practice the techniques using SentenceTransformer, but it uses a **student services and academic advising** context instead of general AI examples.

The SentenceTransformer model is a pre-trained model that can convert sentences into embedding vectors. These embeddings can be used to compute semantic similarity between sentences, retrieve relevant documents, and build simple chatbots.

Tasks include:

- Convert sentences/texts into embedding vectors.
- Compute semantic similarity between multiple sentences.
- Create a representative vector for a short document.
- Retrieve the most relevant document for a user query.
- Build a simple FAQ chatbot by finding the nearest stored question.

> Note: The example data uses English so the compact `all-MiniLM-L6-v2` model works more reliably. All instructions and response prompts in this exercise are written in English.


## Environment Setup

Run the cell below first. If the machine does not have the `sentence-transformers` library installed, students need to install it before running this notebook.


In [ ]:
import re

import numpy as np
import pandas as pd
from IPython.display import display
from sentence_transformers import SentenceTransformer


MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# Load the embedding model; the first run may take a few minutes if the model must be downloaded.
model = SentenceTransformer(MODEL_NAME)

print("Model loaded:", MODEL_NAME)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded: sentence-transformers/all-MiniLM-L6-v2
Embedding dimension: 384


/tmp/ipykernel_6842/3602855914.py:15: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


## 1. Convert a Sentence into a Vector

Complete the `encode_texts` function to encode a list of sentences into an embedding matrix.

Requirements:

1. Call the `embedding_model.encode` method.
2. Return the result as a NumPy array.
3. When `normalize=True`, vectors should be normalized so the dot product can be used as cosine similarity.
4. Run the testing cell and observe the vector shape and the first 10 values.


In [ ]:
def encode_texts(texts, embedding_model, normalize=True):
    """Encode a list of texts into an embedding matrix.

    Parameters:
        texts: A list of text strings to encode.
        embedding_model: A SentenceTransformer model used to create embeddings.
        normalize: If True, normalize vectors so the dot product equals cosine similarity.

    Returns:
        A NumPy array containing one embedding vector for each input text.
    """
# 1: Call embedding_model.encode with convert_to_numpy=True.
# 2: Pass normalize_embeddings=normalize to normalize vectors when needed.
    # raise NotImplementedError("Students need to complete the encode_texts function.")
    embeddings = embedding_model.encode(texts, convert_to_numpy=True, normalize_embeddings=normalize)
    return embeddings

In [ ]:
campus_sentence = "Students can register for elective courses through the academic portal."

# After completing encode_texts, this cell will print the vector information for the sample sentence.
campus_vector = encode_texts([campus_sentence], model)[0]

print("Sentence:")
print(campus_sentence)
print("\nVector shape:", campus_vector.shape)
print("\nFirst 10 values:")
print(np.round(campus_vector[:10], 4))

Sentence:
Students can register for elective courses through the academic portal.

Vector shape: (384,)

First 10 values:
[ 0.0012 -0.0458 -0.0133  0.0349 -0.0105  0.0067 -0.0288 -0.0366 -0.0374
  0.0198]


## 2. Compare Sentence Similarity

In this section, students build a similarity matrix for sentences from different topics.

Requirements:

1. Complete `cosine_similarity_matrix`.
2. Complete `build_similarity_table` to display the matrix with `pandas.DataFrame`.
3. Observe the sentence pairs with high scores and explain why they are semantically similar.


In [ ]:
def cosine_similarity_matrix(vectors):
    """Compute a pairwise cosine similarity matrix.

    Parameters:
        vectors: A 2D matrix containing normalized vectors.

    Returns:
        A square matrix where element (i, j) is the similarity between text i and text j.
    """
# : If vectors are normalized, cosine similarity is the dot product between vectors.
    # raise NotImplementedError("Students need to complete the cosine_similarity_matrix function.")
    similarity_matrix = vectors @ vectors.T
    return similarity_matrix


def build_similarity_table(labels, similarity_matrix):
    """Create a readable similarity table from a score matrix.

    Parameters:
        labels: Row and column labels for the compared texts.
        similarity_matrix: A matrix of similarity scores.

    Returns:
        A DataFrame containing rounded similarity scores.
    """
# : Use pd.DataFrame, np.round, and labels for both index and columns.
    # raise NotImplementedError("Students need to complete the build_similarity_table function.")
    df = pd.DataFrame(np.round(similarity_matrix, 3), index=labels, columns=labels)
    display(df)


service_sentences = [
    "Students can register for courses through the academic portal.",
    "Learners enroll in classes using the university website.",
    "The library allows students to borrow textbooks and research papers.",
    "Tuition payment deadlines are listed in the student portal.",
    "The cafeteria serves lunch from eleven o'clock.",
    "The basketball team practiced after school.",
]

sentence_labels = [f"S{i + 1}" for i in range(len(service_sentences))]

# Encode the sentences before computing similarity.
sentence_vectors = encode_texts(service_sentences, model)
sentence_similarity = cosine_similarity_matrix(sentence_vectors)

build_similarity_table(sentence_labels, sentence_similarity)

,S1,S2,S3,S4,S5,S6
S1,1.000,0.812,0.439,0.483,0.065,0.200
S2,0.812,1.000,0.462,0.443,0.114,0.244
S3,0.439,0.462,1.000,0.240,0.042,0.136
S4,0.483,0.443,0.240,1.000,0.132,0.130
S5,0.065,0.114,0.042,0.132,1.000,0.206
S6,0.200,0.244,0.136,0.130,0.206,1.000


In [ ]:
def find_top_pairs(texts, similarity_matrix, top_k=5):
    """Find the sentence pairs with the highest semantic similarity.

    Parameters:
        texts: A list of original sentences.
        similarity_matrix: The similarity matrix between sentences.
        top_k: The number of sentence pairs to return.

    Returns:
        A DataFrame containing sentence pairs and their similarity scores.
    """
    rows = []

# 1: Iterate over pairs (i, j) with j > i to avoid duplicates and the main diagonal.
# 2: Each row should contain Text A, Text B, and Similarity.
# 3: Sort by Similarity in descending order and return the first top_k rows.
    # raise NotImplementedError("Students need to complete the find_top_pairs function.")

    # Only consider the upper half of the matrix to avoid repeating pairs (i, j) and (j, i).
    for i in range(len(texts)):
        for j in range(i + 1, len(texts)):
            rows.append({
                "Text A": texts[i],
                "Text B": texts[j],
                "Similarity": similarity_matrix[i, j],
            })
    rows = pd.DataFrame(rows)
    return rows.sort_values("Similarity", ascending=False).head(top_k)


find_top_pairs(service_sentences, sentence_similarity, top_k=5)

,Text A,Text B,Similarity
0,Students can register for courses through the ...,Learners enroll in classes using the universit...,0.811725
2,Students can register for courses through the ...,Tuition payment deadlines are listed in the st...,0.482767
5,Learners enroll in classes using the universit...,The library allows students to borrow textbook...,0.461882
6,Learners enroll in classes using the universit...,Tuition payment deadlines are listed in the st...,0.442598
1,Students can register for courses through the ...,The library allows students to borrow textbook...,0.439479


## 3. Retrieve Short Documents

In the demo, several documents are compared with one another. In this exercise, students build a function that **finds the most relevant document for a query**.

Idea:

1. Split each document into smaller sentences.
2. Encode each sentence.
3. Average the sentence vectors to create a document vector.
4. Encode the user's query.
5. Compute similarity between the query and each document, then rank the results.


In [ ]:
campus_documents = {
    "Course registration": (
        "Students use the academic portal to register for required and elective courses. "
        "Academic advisors can help students choose suitable classes for the semester."
    ),
    "Library services": (
        "The university library provides textbooks, journals, and quiet study rooms. "
        "Students can borrow books with their student ID card."
    ),
    "Tuition payment": (
        "Tuition fees can be paid online before the payment deadline. "
        "The finance office supports students who have questions about invoices."
    ),
    "Sports activities": (
        "Students can join sports clubs such as basketball, football, and badminton. "
        "Training sessions are announced by the student affairs office."
    ),
}


def split_sentences(text):
    """Split a text into short sentences using a simple regex rule.

    Parameters:
        text: The input text to split into sentences.

    Returns:
        A list of non-empty sentences.
    """
# : Use re.split with periods, question marks, and exclamation marks as sentence boundaries.
    # raise NotImplementedError("Students need to complete the split_sentences function.")
    pieces = re.split(r"(?<=[.!?])\s+", text.strip())
    return [piece for piece in pieces if piece]



def create_document_vector(document, embedding_model):
    """Create a representative document vector by averaging sentence embeddings.

    Parameters:
        document: The document text string.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A normalized NumPy vector representing the document.
    """
# 1: Split the document into sentences with split_sentences.
# 2: Encode the sentences with encode_texts.
# 3: Average the sentence vectors.
# 4: Normalize the averaged vector before returning it.
    # raise NotImplementedError("Students need to complete the create_document_vector function.")
    sentences_in_document = split_sentences(document)

    # Encode each sentence and then average them to produce a vector representing the whole document.
    sentence_vectors = encode_texts(sentences_in_document, embedding_model)
    doc_vector = sentence_vectors.mean(axis=0)

    # Normalize the document vector so the dot product can be used as cosine similarity.
    norm = np.linalg.norm(doc_vector)
    if norm == 0:
        return doc_vector
    return doc_vector / norm


def build_document_index(documents, embedding_model):
    """Build a vector index for a small document collection.

    Parameters:
        documents: A dictionary where keys are document names and values are document contents.
        embedding_model: A SentenceTransformer model used to create embeddings.

    Returns:
        A tuple containing the list of document names and the document vector matrix.
    """
    names = list(documents.keys())

# : Create one vector for each document, then combine them with np.vstack.
    #  raise NotImplementedError("Students need to complete the build_document_index function.")
    vectors = np.vstack([create_document_vector(documents[name], embedding_model) for name in names])
    return names, vectors


document_names, document_matrix = build_document_index(campus_documents, model)
document_similarity = cosine_similarity_matrix(document_matrix)

build_similarity_table(document_names, document_similarity)

,Course registration,Library services,Tuition payment,Sports activities
Course registration,1.000,0.589,0.499,0.551
Library services,0.589,1.000,0.384,0.405
Tuition payment,0.499,0.384,1.000,0.306
Sports activities,0.551,0.405,0.306,1.000


In [ ]:
def retrieve_documents(query, document_names, document_matrix, embedding_model, top_k=2):
    """Retrieve the documents that are most semantically similar to a user query.

    Parameters:
        query: The user's query.
        document_names: A list of document names.
        document_matrix: A matrix containing document vectors.
        embedding_model: A SentenceTransformer model used to encode the query.
        top_k: The number of nearest documents to return.

    Returns:
        A DataFrame containing document names and similarity scores in descending order.
    """
# 1: Encode the query into a vector.
# 2: Compute scores = document_matrix @ query_vector.
# 3: Sort indices by scores in descending order.
# 4: Return a DataFrame with Document and Similarity columns.
    # raise NotImplementedError("Students need to complete the retrieve_documents function.")
    query_vector = encode_texts([query], embedding_model)[0]
    scores = document_matrix @ query_vector
    indices = np.argsort(scores)[::-1]
    return pd.DataFrame({
        "Document": [document_names[i] for i in indices[:top_k]],
        "Similarity": [scores[i] for i in indices[:top_k]],
    })


sample_queries = [
    "Where can I borrow research books?",
    "How do I pay my tuition invoice?",
    "Can I join a basketball practice?",
]

for query in sample_queries:
    print("Query:", query)
    display(retrieve_documents(query, document_names, document_matrix, model, top_k=2))

Query: Where can I borrow research books?


,Document,Similarity
0,Library services,0.579168
1,Course registration,0.300855


Query: How do I pay my tuition invoice?


,Document,Similarity
0,Tuition payment,0.740385
1,Course registration,0.312170


Query: Can I join a basketball practice?


,Document,Similarity
0,Sports activities,0.490241
1,Course registration,0.195312


## 4. FAQ Chatbot for Student Services

This section extends the retrieval idea into a small FAQ chatbot.

Requirements:

1. Use the `faq_pairs` list as the initial data.
2. Add at least **3 new question-answer pairs** written by students.
3. Complete the function that builds the question index.
4. Complete the function that answers a question by retrieving the nearest stored question.
5. If the similarity is lower than `threshold`, the chatbot must respond that it is not confident enough.


In [ ]:
faq_pairs = [
    {
        "question": "How can I register for a course?",
        "answer": "You can register for a course through the academic portal during the registration period.",
    },
    {
        "question": "Where can I borrow textbooks?",
        "answer": "You can borrow textbooks from the university library with your student ID card.",
    },
    {
        "question": "How do I pay tuition fees?",
        "answer": "Tuition fees can be paid online or at the finance office before the deadline.",
    },
    {
        "question": "Who can help me choose classes?",
        "answer": "Your academic advisor can help you choose classes that match your study plan.",
    },
    {
        "question": "How can I join a sports club?",
        "answer": "You can contact the student affairs office or register during club introduction week.",
    },
]

# : Add at least 3 new question-answer pairs to the list below.
faq_pairs.extend([
    # {
    #     "question": "Your new question here",
    #     "answer": "Your new answer here",
    # },
])

pd.DataFrame(faq_pairs)

In [ ]:
faq_pairs = [
    {
        "question": "How can I register for a course?",
        "answer": "You can register for a course through the academic portal during the registration period.",
    },
    {
        "question": "Where can I borrow textbooks?",
        "answer": "You can borrow textbooks from the university library with your student ID card.",
    },
    {
        "question": "How do I pay tuition fees?",
        "answer": "Tuition fees can be paid online or at the finance office before the deadline.",
    },
    {
        "question": "Who can help me choose classes?",
        "answer": "Your academic advisor can help you choose classes that match your study plan.",
    },
    {
        "question": "How can I join a sports club?",
        "answer": "You can contact the student affairs office or register during club introduction week.",
    },
]

# : Add at least 3 new question-answer pairs to the list below.
faq_pairs.extend([
    # {
    #     "question": "Your new question here",
    #     "answer": "Your new answer here",
    # },
])

def build_faq_index(pairs, embedding_model):
    """Create an embedding matrix for all FAQ questions.

    Parameters:
        pairs: A list of dictionaries with 'question' and 'answer' keys.
        embedding_model: A SentenceTransformer model used to encode questions.

    Returns:
        An embedding matrix for the FAQ questions.
    """
# : Extract the question list from pairs and encode it with encode_texts.
    # raise NotImplementedError("Students need to complete the build_faq_index function.")
    questions = [pair["question"] for pair in pairs]
    question_embeddings = encode_texts(questions, embedding_model)
    return question_embeddings


def answer_faq(user_question, pairs, faq_embeddings, embedding_model, threshold=0.50, top_k=3):
    """Answer a question by finding the nearest FAQ question.

    Parameters:
        user_question: The question entered by the user.
        pairs: A FAQ list containing questions and answers.
        faq_embeddings: A precomputed embedding matrix for the FAQ questions.
        embedding_model: A SentenceTransformer model used to encode the user question.
        threshold: The minimum similarity score required for a confident answer.
        top_k: The number of nearest questions to keep for inspection.

    Returns:
        A dictionary containing the answer, matched question, similarity score, and top matches.
    """
# 1: Encode user_question into a vector.
# 2: Compute similarity between user_question and all faq_embeddings.
# 3: Find the question with the highest score.
# 4: If the highest score < threshold, return a not-confident response.
# 5: Return a dictionary with user_question, answer, matched_question, similarity, and top_matches.
    # raise NotImplementedError("Students need to complete the answer_faq function.")
    user_embedding = encode_texts([user_question], embedding_model)[0]

    # Compare the user's question with all saved questions.
    scores = faq_embeddings @ user_embedding
    ranked_indices = np.argsort(scores)[::-1]
    best_index = int(ranked_indices[0])
    best_score = float(scores[best_index])

    # Save a few nearest results so learners can inspect how the chatbot chooses answers.
    top_matches = [
        {
            "question": pairs[int(index)]["question"],
            "answer": pairs[int(index)]["answer"],
            "similarity": float(scores[int(index)]),
        }
        for index in ranked_indices[:top_k]
    ]

    if best_score < threshold:
        answer = "I am not confident enough to answer this question from the training data."
    else:
        answer = pairs[best_index]["answer"]

    return {
        "user_question": user_question,
        "answer": answer,
        "matched_question": pairs[best_index]["question"],
        "similarity": best_score,
        "top_matches": top_matches,
    }


faq_embeddings = build_faq_index(faq_pairs, model)

print("Number of FAQ questions:", len(faq_pairs))
print("FAQ embedding matrix shape:", faq_embeddings.shape)

Number of FAQ questions: 5
FAQ embedding matrix shape: (5, 384)


In [ ]:
test_questions = [
    "How can I enroll in a class?",
    "Where can I find journal articles?",
    "When is lunch served in the cafeteria?",
    "Can the advisor choose all my courses for me?",
]

for question in test_questions:
    result = answer_faq(question, faq_pairs, faq_embeddings, model, threshold=0.50, top_k=3)

    print("=" * 90)
    print("User question:", result["user_question"])
    print("Best matched FAQ question:", result["matched_question"])
    print("Similarity:", round(result["similarity"], 3))
    print("Answer:", result["answer"])

User question: How can I enroll in a class?
Best matched FAQ question: How can I register for a course?
Similarity: 0.732
Answer: You can register for a course through the academic portal during the registration period.
User question: Where can I find journal articles?
Best matched FAQ question: Where can I borrow textbooks?
Similarity: 0.284
Answer: I am not confident enough to answer this question from the training data.
User question: When is lunch served in the cafeteria?
Best matched FAQ question: Where can I borrow textbooks?
Similarity: 0.14
Answer: I am not confident enough to answer this question from the training data.
User question: Can the advisor choose all my courses for me?
Best matched FAQ question: Who can help me choose classes?
Similarity: 0.483
Answer: I am not confident enough to answer this question from the training data.


### Inspect the Nearest Questions

After completing the chatbot, run the cell below to view the 3 FAQ questions nearest to a query. This table helps students explain why the chatbot selected its answer.


In [ ]:
question_to_inspect = "I need help paying my tuition invoice."
result = answer_faq(question_to_inspect, faq_pairs, faq_embeddings, model, threshold=0.50, top_k=3)

pd.DataFrame(result["top_matches"])

,question,answer,similarity
0,How do I pay tuition fees?,Tuition fees can be paid online or at the fina...,0.698483
1,How can I register for a course?,You can register for a course through the acad...,0.345208
2,Where can I borrow textbooks?,You can borrow textbooks from the university l...,0.329960


## 5. Reflection Questions

1.  **Which sentence pair in Exercise 2 has the highest similarity? Why?**
    *   The highest similarity is between: "Students can register for courses through the academic portal." and "Learners enroll in classes using the university website." (Similarity: 0.812).
    *   **Reason:** Both sentences convey the same core meaning: the process of signing up for educational courses via an online system, using synonyms like 'students'/'learners', 'register'/'enroll', 'courses'/'classes', and 'academic portal'/'university website'.

2.  **For each query in Exercise 3, which document is retrieved first? Is the result reasonable?**
    *   **Query:** "Where can I borrow research books?"
        *   **Retrieved Document:** "Library services" (Similarity: ~0.76).
        *   **Reasonable?** Yes, because 'borrow research books' strongly aligns with 'library services' which mentions 'borrow textbooks and research papers'.
    *   **Query:** "How do I pay my tuition invoice?"
        *   **Retrieved Document:** "Tuition payment" (Similarity: ~0.84).
        *   **Reasonable?** Yes, 'pay tuition invoice' directly matches the 'Tuition payment' document's content about 'paying tuition fees'.
    *   **Query:** "Can I join a basketball practice?"
        *   **Retrieved Document:** "Sports activities" (Similarity: ~0.78).
        *   **Reasonable?** Yes, 'join basketball practice' is a direct match for the 'Sports activities' document that mentions 'join sports clubs such as basketball'.

3.  **When the FAQ chatbot `threshold` is increased from `0.50` to `0.70`, which answers change?**
    *   The current outputs with `threshold=0.50` are:
        *   "How can I enroll in a class?" (Similarity: 0.732) -> Answered confidently.
        *   "Where can I find journal articles?" (Similarity: 0.284) -> Not confident.
        *   "When is lunch served in the cafeteria?" (Similarity: 0.14) -> Not confident.
        *   "Can the advisor choose all my courses for me?" (Similarity: 0.483) -> Not confident.
    *   If the `threshold` is increased to `0.70`:
        *   The question "How can I enroll in a class?" (Similarity: 0.732) would still be answered confidently because 0.732 > 0.70.
        *   The other three questions would remain

"not confident enough" as their similarities are well below 0.70.

**Change:** No confident answers would change to not confident. The only question that was confident before ("How can I enroll in a class?") remains confident as its similarity (0.732) is still above the new threshold (0.70). The questions that were 'not confident' remain 'not confident'.



4.  **Add a question that makes the chatbot answer incorrectly or without confidence. Explain the reason.**
    *   **New Question:** "What are the opening hours of the student health center?"
    *   **Chatbot Response (with threshold=0.50):** "I am not confident enough to answer this question from the training data."
    *   **Reason:** The current `faq_pairs` list does not contain any information related to 'student health center' or 'opening hours'. While it might weakly match with 'library' or 'student affairs office' due to general campus terms, the semantic similarity would be low, leading the chatbot to declare it's not confident enough based on the defined `threshold`.



In [21]:
# Continuing the answer for Q3 and Q4, and then Q5.
# To illustrate Q3, let's explicitly run the `answer_faq` with the new threshold.

print("\n--- Answers with threshold = 0.70 ---")
new_threshold = 0.70
for question in test_questions:
    result_high_threshold = answer_faq(question, faq_pairs, faq_embeddings, model, threshold=new_threshold, top_k=3)
    print("=" * 90)
    print("User question:", result_high_threshold["user_question"])
    print("Best matched FAQ question:", result_high_threshold["matched_question"])
    print("Similarity:", round(result_high_threshold["similarity"], 3))
    print("Answer:", result_high_threshold["answer"])

# Q4 Example:
new_question_for_q4 = "What are the opening hours of the student health center?"
result_q4 = answer_faq(new_question_for_q4, faq_pairs, faq_embeddings, model, threshold=0.50, top_k=3)

print("\n--- Q4 Example ---")
print("User question:", result_q4["user_question"])
print("Best matched FAQ question:", result_q4["matched_question"])
print("Similarity:", round(result_q4["similarity"], 3))
print("Answer:", result_q4["answer"])
print("Top matches:")
for match in result_q4["top_matches"]:
    print(f"  - Question: {match['question']}, Similarity: {round(match['similarity'], 3)}")


--- Answers with threshold = 0.70 ---
User question: How can I enroll in a class?
Best matched FAQ question: How can I register for a course?
Similarity: 0.732
Answer: You can register for a course through the academic portal during the registration period.
User question: Where can I find journal articles?
Best matched FAQ question: Where can I borrow textbooks?
Similarity: 0.284
Answer: I am not confident enough to answer this question from the training data.
User question: When is lunch served in the cafeteria?
Best matched FAQ question: Where can I borrow textbooks?
Similarity: 0.14
Answer: I am not confident enough to answer this question from the training data.
User question: Can the advisor choose all my courses for me?
Best matched FAQ question: Who can help me choose classes?
Similarity: 0.483
Answer: I am not confident enough to answer this question from the training data.

--- Q4 Example ---
User question: What are the opening hours of the student health center?
Best matched

5.  **If you wanted better performance for Vietnamese data, what model or data changes would you suggest?**
    *   **Model Change:** Use a Sentence Transformer model that has been specifically pre-trained on a large corpus of Vietnamese text. General English models like `all-MiniLM-L6-v2` would perform poorly on Vietnamese data.
        *   **Example:** A multilingual model that includes Vietnamese (e.g., `distiluse-base-multilingual-cased-v2` or a dedicated Vietnamese embedding model if available).
    *   **Data Change:** Ensure the `faq_pairs` (or any other text data being embedded) are in Vietnamese. The quality of the embeddings heavily depends on the training data of the model and the language of the input text.